# PN22 — odd-lattice ARA candidate test

## tl;dr

`T(A)=oddceil(7A/2+1)` is a coherent odd-compatible construction. Across one million inputs it produced a `16.6740%` prime rate versus `14.2942%` for raw odds, but its candidates exactly equal four fixed residue lanes `{1,5,9,13} mod 14`. Its matched-control lift is exactly `1.0`: an exact wheel-sieve crosswalk, not a new prime locator.


## Context & Methods

The continuous ARA identity uses `A`, `B=2A`, ridge offset `A/2`, and a closing `+1`. `oddceil` projects upward to the first allowed odd integer. Inputs are every `A=1,...,1,000,000`; exact prime labels come from Eratosthenes.

### Key Assumptions

- Upward projection is fixed before outcomes are observed.
- Raw odd, coprime-to-14 and exact-residue controls separate lattice filtering from prime-specific information.
- Perfect powers are a predeclared secondary subgroup because 27 and 32 are both perfect powers.


In [1]:
import json
from pathlib import Path
import numpy as np
from pn22_odd_lattice_ara_candidate import oddceil_transform, prime_flags, MAX_A

HERE = Path.cwd()
saved = json.loads((HERE / 'PN22_ODD_LATTICE_ARA_CANDIDATE_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN22_ODD_LATTICE_ARA_CANDIDATE_VALIDATION.json').read_text(encoding='utf-8'))
print('Loaded frozen result and independent validation.')


Loaded frozen result and independent validation.


## Data

Generate the integer inputs, projected candidates and exact prime table.


In [2]:
A = np.arange(1, MAX_A + 1, dtype=np.int64)
T = oddceil_transform(A)
flags = prime_flags(int(T.max()) + 140)
print('inputs:', A.size)
print('output range:', int(T.min()), 'to', int(T.max()))
print('unique outputs:', np.unique(T).size)
assert np.unique(T).size == A.size


inputs: 1000000
output range: 5 to 3500001
unique outputs: 1000000


## Results

First verify the four piecewise branches and their exact residue lanes.


In [3]:
for remainder in range(4):
    sample = A[A % 4 == remainder][:3]
    print('A mod 4 =', remainder, 'samples:', list(zip(sample.tolist(), oddceil_transform(sample).tolist())))
residues = sorted(set((T % 14).tolist()))
print('output residues mod 14:', residues)
assert residues == [1, 5, 9, 13]


A mod 4 = 0 samples: [(4, 15), (8, 29), (12, 43)]
A mod 4 = 1 samples: [(1, 5), (5, 19), (9, 33)]
A mod 4 = 2 samples: [(2, 9), (6, 23), (10, 37)]
A mod 4 = 3 samples: [(3, 13), (7, 27), (11, 41)]
output residues mod 14: [1, 5, 9, 13]


In [4]:
low, high = int(T.min()), int(T.max())
output_range = np.arange(low, high + 1, dtype=np.int64)
odd = output_range[(output_range & 1) == 1]
coprime14 = output_range[np.gcd(output_range, 14) == 1]
matched = output_range[np.isin(output_range % 14, [1, 5, 9, 13])]
rates = {
    'ARA candidates': float(flags[T].mean()),
    'raw odds': float(flags[odd].mean()),
    'coprime to 14': float(flags[coprime14].mean()),
    'exact matched lanes': float(flags[matched].mean()),
}
for name, rate in rates.items():
    print(name, f'{100*rate:.6f}%')
assert np.array_equal(T, matched)
assert rates['ARA candidates'] == rates['exact matched lanes']


ARA candidates 16.674000%
raw odds 14.294180%
coprime to 14 16.676478%
exact matched lanes 16.674000%


In [5]:
print('worked examples')
for row in saved['examples']:
    print(row['A'], '->', row['T_A'], 'prime' if row['T_A_is_prime'] else 'composite')
print('perfect-power candidate rate:', saved['subgroups']['all_unique_perfect_powers']['candidate_prime_rate'])
print('perfect-power matched control:', saved['subgroups']['all_unique_perfect_powers']['matched_local_control_prime_rate'])
assert saved['decision']['wheel_crosswalk'] is True
assert saved['decision']['blind_target_authorized'] is False
print('independent validation:', validation['status'], validation['checks_passed'], '/', validation['checks_total'])
assert validation['status'] == 'PASS'


worked examples
27 -> 97 prime
32 -> 113 prime
34 -> 121 composite
36 -> 127 prime
28 -> 99 composite
30 -> 107 prime
40 -> 141 composite
48 -> 169 composite
52 -> 183 composite
56 -> 197 prime
perfect-power candidate rate: 0.10720720720720721
perfect-power matched control: 0.18092357694217487
independent validation: PASS 16 / 16


## Takeaways

1. Half-integer ridges from odd inputs are handled coherently by a predeclared upward odd-lattice projection.
2. The transformation exactly avoids the factor-2 and factor-7 schedules, giving a real 16.65% lift over raw odds.
3. It contains no enrichment beyond its exact modulo-14 lanes, and perfect-power inputs underperformed same-lane local controls.
4. An additional independently defined ARA term would be required to handle remaining factor collisions before any fresh prime prediction.
